In [1]:
import pandas as pd
import numpy as np

from sksurv.ensemble import RandomSurvivalForest
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sksurv.util import Surv

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sksurv.linear_model import CoxnetSurvivalAnalysis


In [2]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    nacc_data_csv = "/content/drive/MyDrive/bachelor/nacc_data_2025.csv"
else:
    nacc_data_csv = "nacc_data_2025.csv"

In [3]:
nacc_raw = pd.read_csv(nacc_data_csv, delimiter='\t')

/var/folders/g9/j9c8b7vs3m3frqcxprr6hf900000gn/T/ipykernel_48489/2964923166.py:1: DtypeWarning: Columns (8,10,12,14,25,29,31,40,42,44,46,48,50,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,142,194,197,199,205,207,209,211,213,215,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,374,376,378,396,398,409,422,429,469,549,572,580,605,640,673,676,693,704,710,763,765,766,767,768,774,797,809,810,818,819,820,821,831,853,856,859) have mixed types. Specify dtype option on import or set low_memory=False.
  nacc_raw = pd.read_csv(nacc_data_csv, delimiter='\t')


# Helper functions

These function were created and implemented suring EDA phase, and they will be used for the further preprocessing

In [4]:
def filter_columns_by_missing_pattern(df, reference_col='HIV'):
    print(f"Filtering columns by missing pattern")
    
    if reference_col not in df.columns:
        raise KeyError(f"Column '{reference_col}' not found in dataframe.")

    reference_mask = df[reference_col].isna().to_numpy()

    forward_columns = []
    opposite_columns = []

    for col in df.columns:
        col_mask = df[col].isna().to_numpy()

        if np.array_equal(col_mask, reference_mask):
            forward_columns.append(col)
        elif np.array_equal(col_mask, ~reference_mask):
            opposite_columns.append(col)

    kept_columns = forward_columns + opposite_columns
    dropped_columns = [col for col in df.columns if col not in kept_columns]
    filtered_df = df[kept_columns].copy()

    return filtered_df, forward_columns, opposite_columns, dropped_columns

In [5]:
LOW_MISSINGNESS_THRESHOLD = 20
HIGH_MISSINGNESS_THRESHOLD = 80

In [6]:
def define_missingnes(df):
  print(f"Defining missingness")
  missing_percantage_per_column = df.isna().sum() / len(df) * 100

  low_missing = missing_percantage_per_column[missing_percantage_per_column < LOW_MISSINGNESS_THRESHOLD].index.tolist()
  medium_missing = missing_percantage_per_column[(missing_percantage_per_column >= LOW_MISSINGNESS_THRESHOLD) & (missing_percantage_per_column < HIGH_MISSINGNESS_THRESHOLD)].index.tolist()
  high_missing = missing_percantage_per_column[missing_percantage_per_column >= HIGH_MISSINGNESS_THRESHOLD].index.tolist()

  return low_missing, medium_missing, high_missing

In [7]:
NOT_COLLECTED_PLACEHOLDER_VALUE = -99.0
important_very_imbalanced_columns = ["NACCFADM", "ELAT", "GAMES", "MOGAIT", "MOSLOW", "BRNINJ", "OTHPSY"]

In [8]:
def define_categorical_and_continuous_columns(df):
    result_df = df.copy()
    unusual_categorical = ['NACCBEHF']
    categorical_cols = []
    continuous_cols = []
    categorical_cols_by_unique_count = {}

    for col in result_df.columns:
        if col == 'EVENT_MCI':
            continue
        
        n_unique = result_df[col].nunique(dropna=True)
        if n_unique == 1:
            result_df.drop(columns=[col], inplace=True)
            continue
        if 2 <= n_unique <= 10 or col in unusual_categorical:
            categorical_cols.append(col)
            if n_unique not in categorical_cols_by_unique_count:
                categorical_cols_by_unique_count[n_unique] = []
            categorical_cols_by_unique_count[n_unique].append(col)
            continue
        continuous_cols.append(col)

    return categorical_cols, continuous_cols, categorical_cols_by_unique_count

In [9]:
def clean_columns(df):
    print(f"Cleaning columns")
    df_clean = df.copy()
    all_categorical_cols, all_continuous_cols, categorical_cols_by_unique_count = define_categorical_and_continuous_columns(df_clean)

    for col in all_categorical_cols:
        if col not in df_clean.columns:
            continue
        col_value_proportions = df_clean[col].value_counts(normalize=True, dropna=True)
        if col_value_proportions.iloc[0] > 0.99 and col not in important_very_imbalanced_columns:
            print(f"Column '{col}' has very high imbalance ({col_value_proportions.iloc[0]:.2%} of one category); consider to drop it.")
            df_clean.drop(columns=[col], inplace=True)

    for col in all_continuous_cols:
        if col not in df_clean.columns:
            continue
        col_var = df_clean[col].var(numeric_only=True)
        if col_var == 0.0000:
            df_clean.drop(columns=[col], inplace=True)
            continue
        if col_var < 0.01 and col not in important_very_imbalanced_columns:
            print(f"Column '{col}' has very low variance ({col_var:.6f}); consider to drop it.")
            df_clean.drop(columns=[col], inplace=True)

    # keep column lists synchronized with the actually retained dataframe
    filtered_categorical_cols = [col for col in all_categorical_cols if col in df_clean.columns]
    filtered_continuous_cols = [col for col in all_continuous_cols if col in df_clean.columns]

    return df_clean, filtered_categorical_cols, filtered_continuous_cols


In [10]:
class RareCategoryCollapser(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, categorical_cols=None, non_collected_placeholder=None):
        self.threshold = threshold
        self.categorical_cols = categorical_cols
        self.non_collected_placeholder = non_collected_placeholder
        self.rare_categories_ = {}

    def fit(self, X, y=None):
        if self.non_collected_placeholder is None:
            raise ValueError('non_collected_placeholder cannot be None')

        df_result = X.copy()
        self.feature_names_in_ = np.asarray(df_result.columns, dtype=object)
        for col in self.categorical_cols:
            if col not in df_result.columns:
                continue

            value_counts = df_result[col][df_result[col] != self.non_collected_placeholder].value_counts(normalize=True)
            rare_categories = value_counts[value_counts < self.threshold].index.tolist()
            self.rare_categories_[col] = rare_categories

        return self

    def transform(self, X):
        df_result = X.copy()

        for col, rare_categories in self.rare_categories_.items():
            if col in df_result.columns and len(rare_categories) > 0:
                rare_categories_str = ', '.join(map(str, rare_categories))
                df_result[col] = df_result[col].replace(rare_categories, f'{col}_{rare_categories_str}')
        return df_result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features, dtype=object)


In [11]:
class CustomOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, categorical_cols=None, selected_features_subset=None):
        self.categorical_cols = categorical_cols
        self.selected_features_subset = selected_features_subset
        self.encoder = None

    def fit(self, X, y=None):
        df_result = X.copy()
        categorical_df = df_result[self.categorical_cols].astype(str)

        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.encoder.fit(categorical_df)

        feature_names = self.encoder.get_feature_names_out(self.categorical_cols)
        self.keep_columns_indices_ = [
            i for i, name in enumerate(feature_names)
            if not name.endswith(f'_{NOT_COLLECTED_PLACEHOLDER_VALUE}') and not name.endswith(f'_{NOT_COLLECTED_PLACEHOLDER_VALUE:.1f}')
        ]
        self.feature_names_out_ = feature_names[self.keep_columns_indices_]

        return self

    def transform(self, X):
        df_result = X.copy()
        categorical_df = df_result[self.categorical_cols].astype(str)
        encoded_array = self.encoder.transform(categorical_df)

        encoded_array = encoded_array[:, self.keep_columns_indices_]
        return encoded_array

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_out_, dtype=object)


In [12]:
class CustomConstantImputer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=None):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        if self.fill_value is None:
            raise ValueError('fill_value cannot be None')
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        df_result = X.copy()
        df_result.fillna(self.fill_value, inplace=True)
        return df_result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features, dtype=object)


In [13]:
def decode_preprocessed_feature_name(feature_name, categorical_cols, continuous_cols):
    if feature_name.startswith('categorical__'):
        remainder = feature_name.replace('categorical__', '', 1)
        matching = [
            col for col in categorical_cols
            if remainder == col or remainder.startswith(f'{col}_')
        ]
        if len(matching) > 0:
            return max(matching, key=len)
        return remainder

    if feature_name.startswith('continuous__'):
        return feature_name.replace('continuous__', '', 1)

    return feature_name

def select_features_subset(df, feature_names, categorical_cols, continuous_cols):
    raw_features = []
    for feature in feature_names:
        base_name = decode_preprocessed_feature_name(feature, categorical_cols, continuous_cols)

        if base_name in df.columns and base_name not in raw_features:
            raw_features.append(base_name)

    for required_col in ['TIME', 'EVENT_MCI', "group_missing_indicator"]:
        if required_col in df.columns and required_col not in raw_features:
            raw_features.append(required_col)

    return df[raw_features]

In [14]:
# drop columns that are target leakage features
# these columns themselves represent the MCI or do not hold relevant information for prediction
def drop_useless_columns(df, columns_to_drop=[]):
    print(f"Dropping useless columns and columns represented the MCI diagnosis")

    df_result = df.copy()
    df_result.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    df_result.drop(columns=['NACCACTV', 'NACCADMD', 'NACCALZD', 'NACCALZP', 'PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF'], inplace=True, errors='ignore')
    df_result.drop(columns=['NACCMCII', 'NACCNORM', 'COGSTAT', 'VISITDAY', 'VISITYR', 'VISITMO', 'NACCETPR'], inplace=True, errors='ignore')
    df_result.drop(columns=['NACCID'], inplace=True, errors='ignore')
    return df_result

In [15]:
def low_missingness_complete_case_analysis(df, low_missingness_columns):
    print(f"Complete-case analysis on low-missing columns")
    df_result = df.copy()
    low_missing_in_filtered = [c for c in low_missingness_columns if c in df_result.columns]
    if len(low_missing_in_filtered) > 0:
      nacc_missing_free_v2_v3 = df_result.dropna(
        subset=low_missing_in_filtered
      ).copy()
    else:
      nacc_missing_free_v2_v3 = df_result.copy()
    return nacc_missing_free_v2_v3

In [16]:
def create_missingness_indicators(df, column_ref_indicator='HIV'):
    print(f"Creating missingness indicator")
    df_result = df.copy()
    df_result['group_missing_indicator'] = np.where(
        df_result[column_ref_indicator].isna(), 1, 0
    )
    return df_result

# Preprocessing pipeline

In [17]:
def build_preprocessing_pipeline(categorical_columns, continuous_columns, selected_features_subset=None):
  # Categorical pipeline
  categorical_pipeline = Pipeline([
    ('imputer', CustomConstantImputer(fill_value=NOT_COLLECTED_PLACEHOLDER_VALUE)),
    ('rare_collapser', RareCategoryCollapser(
      threshold=0.01,
      categorical_cols=categorical_columns,
      non_collected_placeholder=NOT_COLLECTED_PLACEHOLDER_VALUE,
    )),
    ('encoder', CustomOneHotEncoder(categorical_cols=categorical_columns, selected_features_subset=selected_features_subset)),
  ]).set_output(transform='pandas')

  # Continuous features pipeline
  continious_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
  ]).set_output(transform='pandas')

  columns_preprocessing_pipeline = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_columns),
    ('continuous', continious_pipeline, continuous_columns),
  ]).set_output(transform='pandas')

  return Pipeline([
    ('columns_preprocessing', columns_preprocessing_pipeline),
  ]).set_output(transform='pandas')


create feature selection up to 95 features with rsf vimp

Skontrolovat podmienky na feature selection. Nutne podmienky musia byt splnene, dostatocne podmienky moszu ale nemusia, staci aj bez nich, ale idealne c nimi.

Overit, ci ako feature selection pracuje c kategorickyi premennami.

NACCBEHF, 

# Feature selection

Accordingto rule of thumb, that each feature should contain at least 10 records, we select 95 features to proceed further

In [18]:
def rsf_vimp_feature_selection(
    x_data,
    y_data,
    n_features,
    n_estimators=200,
    n_repeats=8,
    random_state=42,
):
    # rsf = RandomSurvivalForest(
    #     n_estimators=n_estimators,
    #     random_state=random_state,
    #     max_features='sqrt',
    # )
    # rsf.fit(x_data, y_data)

    # importance = permutation_importance(
    #     rsf,
    #     x_data,
    #     y_data,
    #     n_repeats=n_repeats,
    #     random_state=random_state,
    # )

    # feature_importance = pd.DataFrame(
    #     {
    #         'importances_mean': importance['importances_mean'],
    #         'importances_std': importance['importances_std'],
    #     },
    #     index=x_data.columns,
    # )
    # feature_importance['importances_mean_abs'] = np.abs(feature_importance['importances_mean'])
    # feature_importance = feature_importance.sort_values(by='importances_mean_abs', ascending=False)

    # selected_features = feature_importance.head(n_features).index.tolist()

    cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=0.01, alpha_min_ratio=0.01)
    cox_lasso.fit(x_data, y_data)

    coefficients_lasso = pd.DataFrame(cox_lasso.coef_, index=x_data.columns, columns=np.round(cox_lasso.alphas_, 5))

    optimal_alpha_i = -1
    feature_importance = pd.DataFrame({
        # "feature": x_data.columns,
        "importance": coefficients_lasso.iloc[:, optimal_alpha_i],
        "importance_abs": np.abs(coefficients_lasso.iloc[:, optimal_alpha_i])
    })

    selected_features = feature_importance.sort_values(by='importance_abs', ascending=False).head(n_features).index.tolist()

    return selected_features, feature_importance


# Dataset preprocessing - 1st stage

In [ ]:
class DatasetPreprocessor:
  def __init__(self):
    self.type = None
    self.cleaned_initial_df = None
    self.categorical_columns = None
    self.continuous_columns = None
    self.preprocessing_pipeline = None
    self.N_IMPORTANT_FEATURES = 94

  def define_X_and_y(self, df):    
    X_data = df.drop(columns=['TIME', 'EVENT_MCI'])
    y_data = Surv.from_dataframe('EVENT_MCI', 'TIME', df)

    return X_data, y_data

  def preprocess(self, type="train", df=None):
    if df is None:
      raise ValueError("DataFrame must be provided for preprocessing.")
    
    # 1. We structurally clean the entered dataset,
    # During this step we remove leakage and useless features, create missing indicator
    # clean columns, by deleting hardly imbalanced categorical features and continious features with low variance
    # Indicate missingness and missingness patern, remain only the columns, that are follows the pattern and delete columns, that have >80% missingness
    self.cleaned_initial_df, low_missing_cols = self.structural_cleanup(df.copy())
    self.type = type

    # 2. Perform complete case analysis on low-missingness columns, which have less than 20% missing
    cleaned_df = low_missingness_complete_case_analysis(self.cleaned_initial_df, low_missing_cols)
    X_data, y_data = self.define_X_and_y(cleaned_df)
    
    # 3. Run preprocessing pipeline
    # Impute data, create OneHot encodings for the categorical values, scale continuous features
    preprocessed_X_data = self.run_preprocessing_pipeline(X_data)
    
    # 4. Select features dataset - subset - according to the rule of thumb to proceed further only with that features
    selected_features, rsf_vimp_importance = rsf_vimp_feature_selection(
      preprocessed_X_data,
      y_data,
      n_features=self.N_IMPORTANT_FEATURES,
    )
    # The missing indicator is important feature, that definetelly must be included in the final dataset
    # Since it plays important role in identification of not_collected values, that were simply imputed with constants
    if 'categorical__group_missing_indicator_0' not in selected_features:
      selected_features += ['categorical__group_missing_indicator_0']
    X_subset = preprocessed_X_data[selected_features]
    subset = select_features_subset(
      self.cleaned_initial_df,
      selected_features,
      self.categorical_columns,
      self.continuous_columns
    )

    # 5. Again perform the complete-case analysis on low missingness columns on the subset of selected features
    # Since we do the complete case analysis on the low-missingness features that were selected
    # that brings new records into the final train dataset
    low_missingness_cols_subset = [col for col in low_missing_cols if col in subset.columns]
    subset_cleaned = low_missingness_complete_case_analysis(subset, low_missingness_cols_subset)
    X_subset, y_subset = self.define_X_and_y(subset_cleaned)

    # 6. Rerun the pipeline on the new dataset to perform transformations, OneHot encoding and imputations again
    X_subset_preprocessed = self.run_preprocessing_pipeline(X_subset)

    X_final_subset = X_subset_preprocessed[selected_features]

    print(f"Selected features: {selected_features}")
    print(f"X_subset shape: {X_subset_preprocessed.shape}")
    print(f"X_subset {X_subset_preprocessed.head()}")
    return X_final_subset, y_subset
    
    
  def structural_cleanup(self, df):
    df_result = drop_useless_columns(df)
    low_missing, medium_missing, high_missing = define_missingnes(df_result)
    nacc_pattern_filtered, forward_cols, opposite_cols, pattern_dropped_cols = filter_columns_by_missing_pattern(
      df_result[medium_missing]
    )

    columns_to_proceed = nacc_pattern_filtered.columns.tolist() + low_missing
    nacc_missing_filtered = df_result[columns_to_proceed].copy()

    nacc_filtered_with_miss_indicator = create_missingness_indicators(nacc_missing_filtered, column_ref_indicator='HIV')
    nacc_clean, categorical_cols, continuous_cols = clean_columns(nacc_filtered_with_miss_indicator)
    model_categorical_cols = [c for c in categorical_cols if c not in ['EVENT_MCI', 'TIME']]
    model_continuous_cols = [c for c in continuous_cols if c not in ['EVENT_MCI', 'TIME']]
    self.categorical_columns = model_categorical_cols
    self.continuous_columns = model_continuous_cols

    return nacc_clean, low_missing
  
  def run_preprocessing_pipeline(self, X_data):
    active_categorical_cols = [c for c in self.categorical_columns if c in X_data.columns]
    active_continuous_cols = [c for c in self.continuous_columns if c in X_data.columns]

    preprocessing_pipeline = build_preprocessing_pipeline(active_categorical_cols, active_continuous_cols)
    self.preprocessing_pipeline = preprocessing_pipeline

    if self.type == "fit" or self.type == "fit_transform":
      X_processed = preprocessing_pipeline.fit_transform(X_data)
      return X_processed
    
    X_processed = preprocessing_pipeline.transform(X_data)
    return X_processed


## Split to train test

In [20]:
train_df, test_df = train_test_split(
    nacc_raw,
    test_size=0.2,
    random_state=42,
    stratify=nacc_raw['EVENT_MCI'],
)

In [21]:
data_preprocessor = DatasetPreprocessor()
X_train, y_train = data_preprocessor.preprocess(type="fit", df=train_df)

Dropping useless columns and columns represented the MCI diagnosis
Defining missingness
Filtering columns by missing pattern
Creating missingness indicator
Cleaning columns
Column 'AMNDEM' has very high imbalance (99.65% of one category); consider to drop it.
Column 'PCA' has very high imbalance (99.65% of one category); consider to drop it.
Column 'DATSCAN' has very high imbalance (99.05% of one category); consider to drop it.
Column 'FTLDMO' has very high imbalance (99.98% of one category); consider to drop it.
Column 'FTLDNOS' has very high imbalance (99.83% of one category); consider to drop it.
Column 'PREVSTK' has very high imbalance (99.27% of one category); consider to drop it.
Column 'STROKDEC' has very high imbalance (99.24% of one category); consider to drop it.
Column 'STKIMAG' has very high imbalance (99.24% of one category); consider to drop it.
Column 'EPILEP' has very high imbalance (99.85% of one category); consider to drop it.
Column 'NEOPSTAT' has very high imbalance